In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser()


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import shap 
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(str(DATA_DIR / 'AI_UNITE_CRSS_FARS_merged_IMPAIRMENT_ONLY.csv'))

df = df.dropna(subset=['INJ_SEV'])

df['INJ_SEV'] = df['INJ_SEV'].astype(int)
categorical_cols = [
    'PERNOTMVIT','PVH_INVL','PERMVIT','MONTH','DAY_WEEK','YEAR',
    'HARM_EV','MAN_COLL','TYP_INT','REL_ROAD','WRK_ZONE','LGT_COND','WEATHER',
    'DRDISTRACT','DRIMPAIR','SPEC_USE','SEX','REST_USE','REST_MIS',
    'HELM_USE','HELM_MIS','DRINKING','ALC_STATUS',
    'ALC_RES','DRUGS','STR_VEH','LOCATION','VE_FORMS','HIT_RUN','BODY_TYP','TOW_VEH',
    'CARGO_BT','HAZ_INV','EMER_USE','DR_PRES','SPEEDREL','VTRAFWAY',
    'VSURCOND','VISION', 'VSPD_LIM', 'VE_TOTAL', 'PEDS' # , 'AIR_BAG','EJECTION','ATST_TYP',
] 
numerical_cols = ['TRAV_SP', 'AGE']

target_col = 'INJ_SEV'

for col in categorical_cols:
    df[col] = df[col].astype('category')

X = df[numerical_cols + categorical_cols]
y = df[target_col]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
confusion = confusion_matrix(y_test, y_pred)


plt.imshow(confusion, cmap='Blues', interpolation='nearest')
plt.colorbar()
tick_marks = np.arange(2)
thresh = confusion.max() / 2.
for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        plt.text(j, i, format(confusion[i, j]), ha="center", va="center", color="white" if confusion[i, j] > thresh else "black")



In [ ]:
import shap
import matplotlib.pyplot as plt

rf_model = model.named_steps['classifier']
importances = rf_model.feature_importances_

feature_names = numerical_cols + categorical_cols

indices = np.argsort(importances)[::-1]
top_indices = indices[:20]

plt.figure(figsize=(12, 8))
plt.barh(range(len(top_indices)), importances[top_indices])
plt.yticks(range(len(top_indices)), 
[feature_names[i] if i < len(feature_names) else f"feature_{i}" for i in top_indices])
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('rf_importance.png', dpi=300)
plt.show()
